# Where did RF-DETR's solve get slower?

A same-session L4 comparison measured RF-DETR's shapes against the post-release Git baseline `881e5c3`. That baseline is a comparison commit with the benchmark harness, separate from the published package pinned by RF-DETR. This notebook measures five historical revisions on one GPU and repeats the baseline at the end to estimate drift during the session.

| Revision | Change in this revision |
| --- | --- |
| `881e5c3` | Post-release comparison baseline |
| `dc6b06f` | Solve costs on their own device, 64-bit offsets |
| `6f24fe8` | Runtime matrix dimensions, in-kernel row reduction |
| `9c9c43a` | Indices API, validation modes, small-batch routing |
| `19477d9` | Four warps for large shapes |

In [ ]:
import torch

# The Triton kernel needs compute capability 8.0 or newer, so fail here instead of timing the SciPy fallback.
assert torch.cuda.is_available(), "Choose a GPU runtime: Runtime -> Change runtime type"
capability = torch.cuda.get_device_capability()
assert capability >= (8, 0), (
    f"{torch.cuda.get_device_name()} is sm_{capability[0]}{capability[1]}; the Triton backend needs sm_80 or newer"
)
print(torch.cuda.get_device_name(), f"sm_{capability[0]}{capability[1]}", "torch", torch.__version__)

In [ ]:
# Triton's version decides how the kernel compiles, so the result is only comparable between sessions with the same one.
import triton

print("triton", triton.__version__)

## 1. Get the repository

The first run in this kernel creates a private checkout and a results directory. A benchmark subprocess runs the script and solver from the same revision. Run the setup again in this kernel to resume the experiment. Starting a new kernel starts a new experiment with fresh measurements.

In [ ]:
import json
import math
from pathlib import Path
from tempfile import TemporaryDirectory

# A private directory binds reusable results to this kernel and keeps the clone clean.
if "BISECT_SESSION" not in globals():
    BISECT_SESSION = TemporaryDirectory(prefix="torch-hungarian-bisect-")
REPO = Path(BISECT_SESSION.name) / "repo"
if not REPO.exists():
    !git clone -q https://github.com/Borda/torch-hungarian.git {REPO}

In [ ]:
# Full commit IDs make the comparison independent of later changes to branch tips.
BASELINE_SHA = "881e5c3d6caa1f18853c6b5dbbe89a41ff949199"
RUNS = [
    ("baseline", BASELINE_SHA),
    ("64-bit offsets", "dc6b06f3618415f6e465253db40fdafd2339f705"),
    ("runtime dims", "6f24fe812e482336b19f676e71461b47d0277526"),
    ("indices and routing", "9c9c43a06833a7f03ec8afa648d7f5e75f36955b"),
    ("four warps", "19477d98c08dea4b055397cba8880d2fa9789f0d"),
    ("baseline repeat", BASELINE_SHA),
]
PROCESS_ROUNDS = 3
SHAPES = {(batch, 300, tasks) for batch in (52, 208, 624) for tasks in (1, 2, 5, 10, 20, 30)}
LANES = {
    ("scipy", "full"),
    ("public_cuda", "full"),
    ("triton", "full"),
    ("triton", "off"),
    ("triton", "nonfinite_only"),
    ("triton", "infeasibility_flag_only"),
}

## 2. Benchmark each revision

Each invocation runs the same RF-DETR workloads with three fresh processes per case. The harness must exit successfully and every record must match the expected revision, checkout, backend, shape and round before its output is published. Resuming validates the existing result and reruns any invalid artifact. A failed subprocess leaves its diagnostic file at the partial path and the next attempt can regenerate it.

In [ ]:
def validated_records(path: Path, revision: str) -> list[dict]:
    """Read a complete benchmark result from the requested clean checkout."""
    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    expected = {
        (backend, mode, shape, run) for backend, mode in LANES for shape in SHAPES for run in range(PROCESS_ROUNDS)
    }
    observed = set()
    for record in records:
        if not isinstance(record, dict) or record.get("record_type") != "case" or record.get("status") != "measured":
            raise ValueError(f"Non-measured record in {path}")
        source = record["run_metadata"]["source"]
        stats = record["process_statistics"]
        backend = record["backend"]
        median = stats["warm_median_ms"]["median"]
        if (
            source["benchmark_git_revision"] != revision
            or record["run_label"] != REPO.parent.name
            or record["input_dtype"] != "float32"
            or record["process_isolated"] is not True
            or source["benchmark_git_dirty"] != "clean"
            or Path(source["benchmark_path"]) != (REPO / "tests/benchmark.py").resolve()
            or Path(source["assignment_module_origin"])
            != (REPO / "src/torch_linear_assignment/assignment.py").resolve()
            or (backend != "scipy" and record.get("implementation") != "triton")
            or record["parity"].get("exact") is not True
            or stats["status"] != "measured"
            or stats["rounds"] != list(range(PROCESS_ROUNDS))
            or type(median) not in (int, float)
            or not math.isfinite(median)
            or median <= 0
        ):
            raise ValueError(f"Unexpected provenance or incomplete rounds in {path}")
        shape = record["shape"]
        key = (
            backend,
            record["validation_mode"],
            (shape["batch"], shape["workers"], shape["tasks"]),
            record["process_round"],
        )
        if key in observed:
            raise ValueError(f"Duplicate case round in {path}: {key}")
        observed.add(key)
    if observed != expected:
        raise ValueError(f"Unexpected case inventory in {path}: {len(observed)} of {len(expected)} rounds")
    return records

In [ ]:
def run_rfdetr_preset(label: str, revision: str) -> Path:
    """Reuse verified session results or publish a successful replacement."""
    output = REPO.parent / f"bisect-{label.replace(' ', '-')}-{revision[:12]}.jsonl"
    if output.exists():
        try:
            validated_records(output, revision)
        except (ValueError, KeyError, TypeError) as error:
            print(f"Rerunning {label}: {error}")
        else:
            return output

    !git -C {REPO} checkout -q --detach {revision}
    partial = output.with_suffix(".partial.jsonl")
    # A leftover file from an earlier failed run must not pass for this run's result.
    partial.unlink(missing_ok=True)
    !cd {REPO} && env -u TLA_BENCHMARK_SOURCE_PATH -u TLA_BENCHMARK_GIT_REVISION PYTHONPATH={REPO}/src python tests/benchmark.py --preset rfdetr --backends scipy,public_cuda,triton --dtypes float32 --process-rounds {PROCESS_ROUNDS} --expect-implementation triton --label {REPO.parent.name} --output {partial}
    # A failure or interruption before replacement leaves no new reusable result.
    validated_records(partial, revision)
    partial.replace(output)
    return output

In [ ]:
# The final baseline repeat shows how much the endpoint moved during this session.
outputs = {label: run_rfdetr_preset(label, revision) for label, revision in RUNS}

## 3. Compare with the baseline

Start with the public call using full validation, the path RF-DETR uses. The per-shape ratio shows where latency changes. The final baseline repeat gives an endpoint drift check. It cannot rule out transient changes during an intermediate run.

In [ ]:
def warm_medians(path: Path, revision: str, backend: str, validation: str) -> dict[tuple[int, int], float]:
    """Return the median across rounds for each batch and target count."""
    medians = {}
    for record in validated_records(path, revision):
        if (record["backend"], record["validation_mode"]) == (backend, validation):
            shape = record["shape"]
            medians[(shape["batch"], shape["tasks"])] = record["process_statistics"]["warm_median_ms"]["median"]
    return medians

In [ ]:
import pandas as pd

public = {label: warm_medians(outputs[label], revision, "public_cuda", "full") for label, revision in RUNS}
reference = public["baseline"]
ratios = pd.DataFrame(
    {label: {shape: run[shape] / reference[shape] for shape in sorted(reference)} for label, run in public.items()}
)
ratios.index = [f"B={batch} T={tasks}" for batch, tasks in ratios.index]
display(ratios.round(2))

In [ ]:
# Summarize the ratios across shapes while retaining the detail in the preceding table.
display(ratios.median().round(3).rename("median ratio to baseline").to_frame())

## 4. Compare the private adapter

This lane calls the private Triton adapter with validation off. Its elapsed time still includes input preparation, workspace allocation, result conversion and synchronization. Comparing the public and private series shows which latency changes are visible in both. Isolating kernel time would require narrower measurements.

In [ ]:
private = {label: warm_medians(outputs[label], revision, "triton", "off") for label, revision in RUNS}
private_reference = private["baseline"]
adapter_ratios = pd.DataFrame(
    {
        label: {shape: run[shape] / private_reference[shape] for shape in sorted(private_reference)}
        for label, run in private.items()
    }
)
display(adapter_ratios.median().round(3).rename("private adapter median ratio").to_frame())

Use the two tables to choose the revision and shapes to profile. Keep the benchmark JSONL and the GPU, Torch and Triton versions alongside any reported comparison.